## Adding the real SDSS DT for training

The real dataset is missing `galaxy_population` and `spectral_type` feats found in the synth DT. These feats are derivable.

Equation found [here](https://www.kaggle.com/competitions/playground-series-s6e6/discussion/703535)

`spectral_type` = `r - g`

`galaxy_pop` = `u-r`

In [44]:
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import matplotlib.pyplot as platform
import seaborn as sns

import sys
sys.path.append('..')
%load_ext autoreload
%autoreload 2

from utils.helpers import pre_process, get_x_y, train_finish_notification
real_data = pd.read_csv('../data/star_classification(real DT).csv')
data = pd.read_csv('../data/train.csv')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
real_data.head()

,obj_ID,alpha,delta,u,g,r,i,z,run_ID,rerun_ID,cam_col,field_ID,spec_obj_ID,class,redshift,plate,MJD,fiber_ID
0,1.237661e+18,135.689107,32.494632,23.87882,22.27530,20.39501,19.16573,18.79371,3606,301,2,79,6.543777e+18,GALAXY,0.634794,5812,56354,171
1,1.237665e+18,144.826101,31.274185,24.77759,22.83188,22.58444,21.16812,21.61427,4518,301,5,119,1.176014e+19,GALAXY,0.779136,10445,58158,427
2,1.237661e+18,142.188790,35.582444,25.26307,22.66389,20.60976,19.34857,18.94827,3606,301,2,120,5.152200e+18,GALAXY,0.644195,4576,55592,299
3,1.237663e+18,338.741038,-0.402828,22.13682,23.77656,21.61162,20.50454,19.25010,4192,301,3,214,1.030107e+19,GALAXY,0.932346,9149,58039,775
4,1.237680e+18,345.282593,21.183866,19.43718,17.58028,16.49747,15.97711,15.54461,8102,301,3,137,6.891865e+18,GALAXY,0.116123,6121,56187,842


In [46]:
real_data.columns

Index(['obj_ID', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'run_ID',
       'rerun_ID', 'cam_col', 'field_ID', 'spec_obj_ID', 'class', 'redshift',
       'plate', 'MJD', 'fiber_ID'],
      dtype='object')

In [47]:
data.columns

Index(['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
       'spectral_type', 'galaxy_population', 'class'],
      dtype='object')

In [48]:
unique_to_real_dt = list(set(real_data.columns) - set(data.columns))
unique_to_synth_dt = list(set(data.columns) - set(real_data.columns))
unique_to_real_dt, unique_to_synth_dt

(['MJD',
  'field_ID',
  'cam_col',
  'spec_obj_ID',
  'rerun_ID',
  'obj_ID',
  'plate',
  'fiber_ID',
  'run_ID'],
 ['id', 'galaxy_population', 'spectral_type'])

In [49]:
real_data = real_data.drop(
    columns=['field_ID', 'plate', 'MJD', 'obj_ID',
      'spec_obj_ID', 'rerun_ID', 'run_ID', 'fiber_ID', 'cam_col']
      )

In [50]:
real_data.columns

Index(['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'class', 'redshift'], dtype='object')

In [51]:
# VERIFY. Outsource to helper?
real_data['spectral_type'] = pd.cut(
    real_data['r'] - real_data['g'], 
    [-np.inf, -1, -.5, 0, np.inf],
    labels=['M', 'G/K', 'A/F', 'O/B']
    ).astype(str)

In [52]:
real_data['galaxy_population'] = pd.cut(
        real_data['u'] - real_data['r'], 
        [-np.inf, 2.2, np.inf],
        labels=['Blue_Cloud', 'Red_Sequence']
).astype(str)

In [53]:
real_data.head()

,alpha,delta,u,g,r,i,z,class,redshift,spectral_type,galaxy_population
0,135.689107,32.494632,23.87882,22.27530,20.39501,19.16573,18.79371,GALAXY,0.634794,M,Red_Sequence
1,144.826101,31.274185,24.77759,22.83188,22.58444,21.16812,21.61427,GALAXY,0.779136,A/F,Blue_Cloud
2,142.188790,35.582444,25.26307,22.66389,20.60976,19.34857,18.94827,GALAXY,0.644195,M,Red_Sequence
3,338.741038,-0.402828,22.13682,23.77656,21.61162,20.50454,19.25010,GALAXY,0.932346,M,Blue_Cloud
4,345.282593,21.183866,19.43718,17.58028,16.49747,15.97711,15.54461,GALAXY,0.116123,M,Red_Sequence


In [54]:
real_data.dtypes

alpha                float64
delta                float64
u                    float64
g                    float64
r                    float64
i                    float64
z                    float64
class                 object
redshift             float64
spectral_type         object
galaxy_population     object
dtype: object

In [55]:
data.columns   

Index(['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
       'spectral_type', 'galaxy_population', 'class'],
      dtype='object')

## TESTING THE FORMULA

In [56]:
spec_type_test = pd.cut(
    data['r']  - data['g'], 
    [-np.inf, -1, -.5, 0, np.inf],
    labels=['M', 'G/K', 'A/F', 'O/B']
    ).astype(str)

In [57]:
spec_type_test.shape

(577347,)

In [58]:
(spec_type_test == data['spectral_type']).all()

np.True_

In [59]:
galaxy_pop_test = pd.cut(
        data['u'] - data['r'], 
        [-np.inf, 2.2, np.inf],
        labels=['Blue_Cloud', 'Red_Sequence']
    ).astype(str)

(galaxy_pop_test == data['galaxy_population']).all()

np.True_

---

In [60]:
real_data.columns, real_data.shape

(Index(['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'class', 'redshift',
        'spectral_type', 'galaxy_population'],
       dtype='object'),
 (100000, 11))

In [61]:
data.columns, data.shape

(Index(['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
        'spectral_type', 'galaxy_population', 'class'],
       dtype='object'),
 (577347, 12))

In [62]:
full_dt = pd.concat([data, real_data])

In [63]:
full_dt.columns, full_dt.dtypes

(Index(['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
        'spectral_type', 'galaxy_population', 'class'],
       dtype='object'),
 id                   float64
 alpha                float64
 delta                float64
 u                    float64
 g                    float64
 r                    float64
 i                    float64
 z                    float64
 redshift             float64
 spectral_type         object
 galaxy_population     object
 class                 object
 dtype: object)

In [65]:
full_dt[full_dt['id'].isna()] # ids genearted to fit the `data` dt. It's ok since getx_y removes id col

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,NaN,135.689107,32.494632,23.87882,22.27530,20.39501,19.16573,18.79371,0.634794,M,Red_Sequence,GALAXY
1,NaN,144.826101,31.274185,24.77759,22.83188,22.58444,21.16812,21.61427,0.779136,A/F,Blue_Cloud,GALAXY
2,NaN,142.188790,35.582444,25.26307,22.66389,20.60976,19.34857,18.94827,0.644195,M,Red_Sequence,GALAXY
3,NaN,338.741038,-0.402828,22.13682,23.77656,21.61162,20.50454,19.25010,0.932346,M,Blue_Cloud,GALAXY
4,NaN,345.282593,21.183866,19.43718,17.58028,16.49747,15.97711,15.54461,0.116123,M,Red_Sequence,GALAXY
...,...,...,...,...,...,...,...,...,...,...,...,...
99995,NaN,39.620709,-2.594074,22.16759,22.97586,21.90404,21.30548,20.73569,0.000000,M,Blue_Cloud,GALAXY
99996,NaN,29.493819,19.798874,22.69118,22.38628,20.45003,19.75759,19.41526,0.404895,M,Red_Sequence,GALAXY
99997,NaN,224.587407,15.700707,21.16916,19.26997,18.20428,17.69034,17.35221,0.143366,M,Red_Sequence,GALAXY
99998,NaN,212.268621,46.660365,25.35039,21.63757,19.91386,19.07254,18.62482,0.455040,M,Red_Sequence,GALAXY


In [66]:
full_dt = pre_process(full_dt)
X, Y = get_x_y(full_dt)

In [67]:
x_train, x_eval, y_train, y_eval = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

In [30]:
xgb = XGBClassifier(device='cuda', random_state=3, max_depth=7, n_estimators=271, learning_rate=0.2,
                    subsample=1, colsample_bytree=1, min_child_weight=1, gamma=0, reg_alpha=0, reg_lambda=0
                    )

xgb.fit(x_train, y_train, eval_set=[(x_eval, y_eval)], verbose=100)
y_train_pred = xgb.predict(x_train)
y_eval_pred = xgb.predict(x_eval)
bacc_train = balanced_accuracy_score(y_train, y_train_pred)
bacc_eval = balanced_accuracy_score(y_eval, y_eval_pred)


print(f'Train BACC: {bacc_train} ==== Eval BACC: {bacc_eval}')

[0]	validation_0-mlogloss:0.67281


C:\Users\Lenovo\.pyenv\pyenv-win\versions\3.10.11\lib\site-packages\xgboost\training.py:200: UserWarning: [08:35:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Lenovo\.pyenv\pyenv-win\versions\3.10.11\lib\site-packages\xgboost\training.py:200: UserWarning: [08:35:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[100]	validation_0-mlogloss:0.10645
[200]	validation_0-mlogloss:0.10219
[270]	validation_0-mlogloss:0.10153
Train BACC: 0.972744407789157 ==== Eval BACC: 0.9543777654709377


In [ ]:
import json

with open('../best_params.json') as f:
    best_params = json.load(f)['xgboost_kfold']

xgb = XGBClassifier(device='cuda', random_state=3, **best_params, verbosity=1)

xgb.fit(x_train, y_train, eval_set=[(x_eval, y_eval)], verbose=100)
y_train_pred = xgb.predict(x_train)
y_eval_pred = xgb.predict(x_eval)
bacc_train = balanced_accuracy_score(y_train, y_train_pred)
bacc_eval = balanced_accuracy_score(y_eval, y_eval_pred)


print(f'Train BACC: {bacc_train} ==== Eval BACC: {bacc_eval}')

C:\Users\Lenovo\.pyenv\pyenv-win\versions\3.10.11\lib\site-packages\xgboost\training.py:200: UserWarning: [08:55:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Lenovo\.pyenv\pyenv-win\versions\3.10.11\lib\site-packages\xgboost\training.py:200: UserWarning: [08:55:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[0]	validation_0-mlogloss:0.71622
[100]	validation_0-mlogloss:0.10142
[200]	validation_0-mlogloss:0.09969
[300]	validation_0-mlogloss:0.09922
[400]	validation_0-mlogloss:0.09906
[500]	validation_0-mlogloss:0.09899
[600]	validation_0-mlogloss:0.09898
[700]	validation_0-mlogloss:0.09890
[800]	validation_0-mlogloss:0.09891
[900]	validation_0-mlogloss:0.09888
[1000]	validation_0-mlogloss:0.09886
[1100]	validation_0-mlogloss:0.09885
[1200]	validation_0-mlogloss:0.09885
[1300]	validation_0-mlogloss:0.09882
[1400]	validation_0-mlogloss:0.09884
[1500]	validation_0-mlogloss:0.09886
[1600]	validation_0-mlogloss:0.09885
[1694]	validation_0-mlogloss:0.09881
Train BACC: 0.9709280864144819 ==== Eval BACC: 0.9551183302973457


In [69]:
test_dt = pd.read_csv('../data/test.csv')
test_dt_enc =  pre_process(test_dt).drop(columns=['id'])

In [70]:
test_dt.columns, test_dt.dtypes

(Index(['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
        'spectral_type', 'galaxy_population'],
       dtype='object'),
 id                     int64
 alpha                float64
 delta                float64
 u                    float64
 g                    float64
 r                    float64
 i                    float64
 z                    float64
 redshift             float64
 spectral_type         object
 galaxy_population     object
 dtype: object)

In [ ]:
test_submission_pred = xgb.predict(test_dt_enc)
from utils.helpers import gen_submission

gen_submission(test_submission_pred, 'test')


class
GALAXY    162613
QSO        49989
STAR       34833
Name: count, dtype: int64
